# Offset vs inherent disagreement between BMRB depositions, part 1

Splits the inter-deposition difference into a re-referencing offset and an inherent part, per cluster, and writes `decomposition.json`. Next: `binding_vs_corr_rmse.ipynb`, then `offset_vs_inherent_2_distributions.ipynb`.

In [ ]:
import json, os, pickle, itertools
from collections import defaultdict, Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

def set_style(base=14, family="Arial"):
    plt.rcParams.update({
        "font.family": family, "font.size": base,
        "axes.titlesize": base + 2, "axes.labelsize": base,
        "xtick.labelsize": base - 2, "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2, "svg.fonttype": "none",   # keep text as text in SVG
    })
set_style(16)

MAPPER = "../../data"
HERE   = "Figs"
os.makedirs(HERE, exist_ok=True)
RAW_PKL = f"{MAPPER}/rBMRB/bmrb_cs_data_raw_stage2_noph_notemp.pkl"
NOREF   = f"{MAPPER}/rBMRB/bmrb_cs_data_withoffset_noref_v1.json"
OFFSETS = {
    "LACS":  dict(offset=f"{MAPPER}/rBMRB/bmrb_cs_data_withoffset_lacs_v1.json"),
    "PANAV": dict(offset=f"{MAPPER}/rBMRB/bmrb_cs_data_withoffset_panav_v1.json"),
}
colors = {"LACS": "#98C379", "PANAV": "#E06C75"}

ATOM = "N"
MIN_COMMON = 5          # >=5 common residues per pair
DUP_RMSE   = 0.0       # drop near-duplicate re-depositions (inherent ~0 -> not independent)
TEMP_TOL   = 2.5        # K  : same-temperature tolerance
PH_TOL     = 0.2        # pH unit tolerance

In [ ]:
raw = pickle.load(open(RAW_PKL, "rb"))
nr  = json.load(open(NOREF))

def valid_series(d, atom, key):
    vals = d[atom][key]; mask = d.get(atom + "_mask", {}).get(key)
    out = np.array([np.nan if v is None else float(v) for v in vals], dtype=float)
    if mask is not None:
        m = np.array([bool(x) for x in mask], dtype=bool)
        if len(m) == len(out):
            out[~m] = np.nan
    return out

def _num(s):
    try:    return float(str(s).split()[0].split("-")[0])
    except Exception: return np.nan

def cond_of(eid):
    e = raw.get(eid)
    if e is None: return (np.nan, np.nan)
    return (_num(e.get("temperature")), _num(e.get("pH")))

def offset_by_eid(wo):
    out = {}
    for k in wo["entry_ID"]:
        if k not in wo["offset"]: continue
        v = wo["offset"][k].get("N")
        if v is not None:
            out[wo["entry_ID"][k]] = float(v)
    return out

# ---- clusters: sequence -> {bmrb_id: [keys]} with >=2 depositions ----
by_seq = defaultdict(lambda: defaultdict(list))
for k in nr["sequence"]:
    by_seq[nr["sequence"][k]][nr["entry_ID"][k].split("_")[0]].append(k)
clusters = {s: g for s, g in by_seq.items() if len(g) >= 2}

# print(clusters)
# representative key per deposition (most finite N residues)
reps = {}
for s, b2k in clusters.items():
    reps[s] = {b: max(ks, key=lambda k: np.isfinite(valid_series(nr, ATOM, k)).sum())
               for b, ks in b2k.items()}

size_dist = Counter(len(g) for g in clusters.values())
# n_pairs_theoretical = sum(len(g) * (len(g) - 1) // 2 for g in clusters.values())
print(f"clusters (sequences with >=2 depositions): {len(clusters)}")
print(f"cluster-size distribution  (k depositions : n clusters): {dict(sorted(size_dist.items()))}")
# print(f"theoretical pairs  sum C(k,2): {n_pairs_theoretical}")

In [ ]:
# ---- build all pairs with diff + condition-consistency + cluster size ----
def build_pairs():
    rows = []
    for s, rep in reps.items():
        csize = len(rep)
        for ba, bb in itertools.combinations(list(rep), 2):
            ka, kb = rep[ba], rep[bb]
            a, b = valid_series(nr, ATOM, ka), valid_series(nr, ATOM, kb)
            both = np.isfinite(a) & np.isfinite(b)
            if both.sum() < MIN_COMMON:
                continue
            eid_a, eid_b = nr["entry_ID"][ka], nr["entry_ID"][kb]
            (ta, pa), (tb, pb) = cond_of(eid_a), cond_of(eid_b)
            temp_ok = np.isfinite(ta) and np.isfinite(tb) and abs(ta - tb) <= TEMP_TOL
            ph_ok   = np.isfinite(pa) and np.isfinite(pb) and abs(pa - pb) <= PH_TOL
            rows.append(dict(seq=s, csize=csize, eid_a=eid_a, eid_b=eid_b,
                             diff=a[both] - b[both], n_common=int(both.sum()),
                             d_temp=abs(ta - tb), d_ph=abs(pa - pb),
                             temp_ok=bool(temp_ok), ph_ok=bool(ph_ok),
                             consistent=bool(temp_ok and ph_ok)))
    return rows

pairs = build_pairs()
print(f"usable pairs (>= {MIN_COMMON} common N residues): {len(pairs)}")
pc = sum(p["consistent"] for p in pairs)
print(f"  temp-consistent (<= {TEMP_TOL} K)  : {sum(p['temp_ok'] for p in pairs)}")
print(f"  pH-consistent   (<= {PH_TOL})      : {sum(p['ph_ok'] for p in pairs)}")
print(f"  both consistent                    : {pc}  ({pc/len(pairs):.0%})")

# cluster-level consistency
# clu_consistent = 0
# for s, rep in reps.items():
#     eids = [nr["entry_ID"][rep[b]] for b in rep]
#     ts = [cond_of(e)[0] for e in eids]; ps = [cond_of(e)[1] for e in eids]
#     if (np.nanmax(ts) - np.nanmin(ts) <= TEMP_TOL) and (np.nanmax(ps) - np.nanmin(ps) <= PH_TOL):
#         clu_consistent += 1
# print(f"clusters fully T+pH consistent: {clu_consistent}/{len(reps)}")

In [ ]:
# ---- decompose per pair under each method's stored offsets ----
def decompose(pairs, off_eid, method):
    out = []
    for p in pairs:
        oa, ob = off_eid.get(p["eid_a"]), off_eid.get(p["eid_b"])
        if oa is None or ob is None:
            continue
        offset = float(oa - ob)
        corr_rmse  = float(np.sqrt(np.mean((p["diff"] + offset) ** 2)))
        total_rmse = float(np.sqrt(np.mean(p["diff"] ** 2)))
        out.append(dict(eid_a=p["eid_a"], eid_b=p["eid_b"], method=method, seq=p["seq"], csize=p["csize"],
                        consistent=p["consistent"], n_common=p["n_common"],
                        offset=offset, offset_mse=offset ** 2, oa=oa, ob=ob,
                        inherent_mse=corr_rmse ** 2, corr_rmse=corr_rmse,
                        total_rmse=total_rmse))
    return out

rows = []
for method, cfg in OFFSETS.items():
    off_eid = offset_by_eid(json.load(open(cfg['offset'])))
    rows += decompose(pairs, off_eid, method)
D = pd.DataFrame(rows)
D = D[D["corr_rmse"] >= DUP_RMSE].reset_index(drop=True)   # drop near-duplicate re-depositions
print(f"decomposed pair-records (after dropping near-duplicates < {DUP_RMSE} ppm): {len(D)}")
D = D[D.consistent]
import json
with open("../../data/analysis/Fig3/inherent_test/decomposition.json", "w") as f:
    json.dump(D.to_dict(orient="records"), f, indent=2)
print(f"decomposed pair-records (after dropping inconsistent T+pH): {len(D)}")

In [ ]:
# ===== Fig. S6a: cluster sizes =====
set_style(12)
fig, ax = plt.subplots(figsize=(6, 4.8))

ks = sorted(size_dist)
ax.bar([str(k) for k in ks], [size_dist[k] for k in ks], color="#61AFEF", edgecolor="white")
for i, k in enumerate(ks):
    ax.text(i, size_dist[k], f"{size_dist[k]}\n({k*(k-1)//2*size_dist[k]} pr)",
            ha="center", va="bottom", fontsize=13, font="Arial")
ax.set_xlabel("depositions per sequence cluster")
ax.set_ylabel("number of clusters")
ax.set_ylim(0, max(size_dist.values()) * 1.4)
ax.set_title(f"{len(clusters)} clusters -> {len(pairs)} pairs")
# ax.grid(axis="y", alpha=0.3, ls="--")

plt.tight_layout()
plt.savefig(f"{HERE}/cluster_sizes.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{HERE}/cluster_sizes.svg", bbox_inches="tight")
plt.show()

# ===== Fig. S6b: condition consistency of pairs =====
set_style(12)
fig, ax = plt.subplots(figsize=(6, 4.8))

cats = ["temp\n(<=%.1f K)" % TEMP_TOL, "pH\n(<=%.1f)" % PH_TOL, "both"]
vals = [sum(p["temp_ok"] for p in pairs), sum(p["ph_ok"] for p in pairs),
        sum(p["consistent"] for p in pairs)]
ax.bar(cats, vals, color=["#E5C07B", "#56B6C2", "#98C379"], edgecolor="white")
for i, v in enumerate(vals):
    ax.text(i, v, f"{v}\n({v/len(pairs):.0%})", ha="center", va="bottom", fontsize=13, font="Arial")
# ax.axhline(len(pairs), color="k", ls="--", lw=1, label=f"all pairs ({len(pairs)})")
ax.set_ylabel("number of pairs")
ax.set_ylim(0, max(size_dist.values()) * 1.4)
ax.set_title("condition consistency of pairs")
ax.legend(frameon=False, loc="best")
# ax.grid(axis="y", alpha=0.3, ls="--")

plt.tight_layout()
plt.savefig(f"{HERE}/condition_consistency.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{HERE}/condition_consistency.svg", bbox_inches="tight")
plt.show()